# Videovigilancia con capacidades de IA en los pliegos de contratación pública

Cuaderno de lectura de la medición `videovigilancia/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Licitaciones de videovigilancia de la Plataforma de Contratación del Sector Público entre enero de 2024 y el 12 de octubre de 2026, con el reconocimiento facial, la búsqueda por apariencia, la analítica de vídeo y la lectura de matrículas en el título y en el pliego técnico. Carga los ficheros de al lado (o los descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no repite la medición: para eso está `generar.py`.

*Reading notebook for this measurement: loads the files next to it, prints the provenance record and draws one figure. It only reads; `generar.py` repeats the measurement.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/videovigilancia/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
cap = leer("salidas/conteos_capacidad_nivel.csv")
claves = ["facial_estricto_afirm", "apariencia_estricta_afirm", "analitica_ia_afirm", "lpr_afirm"]
print(cap[(cap.universo == "todas") & cap.capacidad.isin(claves)]
      .pivot(index="capacidad", columns="nivel", values=["n", "denominador", "pct"]).to_string())
print()
lic = leer("salidas/licitaciones_videovigilancia.csv", low_memory=False)
con_texto = lic[lic.estado_pliego.astype(str).str.startswith("ok")].copy()
con_texto["anio"] = con_texto.fecha_publicacion.astype(str).str[:4]
local = con_texto[con_texto.tipo_efectivo.astype(str) == "3"]
por_anio = local.groupby("anio").agg(
    pliegos=("clave", "size"),
    facial=("b_facial_estricto_afirm", "sum"),
    apariencia=("b_apariencia_estricta_afirm", "sum"))
print(por_anio.to_string())
print()
fl = leer("salidas/facial_contra_lista.csv")
print(fl.b_facial_clase.value_counts().to_string())

## Una figura

In [ ]:
pct = por_anio[["facial", "apariencia"]].div(por_anio.pliegos, axis=0) * 100
fig, ax = plt.subplots(figsize=(8, 4))
for col, rotulo in [("facial", "reconocimiento facial"), ("apariencia", "búsqueda por apariencia")]:
    ax.plot(pct.index, pct[col], marker="o", label=rotulo)
    for x, y in zip(pct.index, pct[col]):
        ax.annotate(f"{y:.1f} %", (x, y), textcoords="offset points", xytext=(0, 7), ha="center")
ax.set_ylabel("% de pliegos municipales con texto")
ax.set_ylim(0, 18)
ax.set_title("Qué piden los pliegos de videovigilancia de las entidades locales")
ax.legend(frameon=False)
plt.tight_layout()